# 02. Tiền xử lý và chia dữ liệu

Notebook này chạy lại các bước xử lý dữ liệu bằng các module chính thức trong `src/data/`. Dữ liệu trong `data/raw/` chỉ được đọc, không bị sửa trực tiếp.

Kết quả cần có là `merged.csv`, `cleaned.csv`, `train.csv`, `valid.csv` và `test.csv`. Việc chia dữ liệu được thực hiện trước khi fit encoder hoặc scaler để tránh rò rỉ dữ liệu.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.load_data import save_merged_data
from src.data.clean_data import save_cleaned_data
from src.data.split_data import save_splits

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Bước 1: gộp hai file raw thành một file trung gian.
merged_df = save_merged_data()
print(f'merged.csv: {merged_df.shape[0]} dòng x {merged_df.shape[1]} cột')
display(merged_df.head(3))

# Bước 2: chuẩn hóa kiểu dữ liệu, kiểm tra giá trị và loại dòng trùng.
cleaned_df, duplicate_count = save_cleaned_data()
print(f'cleaned.csv: {cleaned_df.shape[0]} dòng x {cleaned_df.shape[1]} cột')
print(f'Số dòng trùng đã loại: {duplicate_count}')

# Bước 3: chia train/validation/test với random_state cố định.
train_df, valid_df, test_df = save_splits()
split_sizes = pd.Series({
    'train': len(train_df),
    'valid': len(valid_df),
    'test': len(test_df)
})
display(split_sizes.to_frame('rows'))

In [ ]:
# Kiểm tra schema và chất lượng sau tiền xử lý.
expected_first_columns = ['subject', 'school', 'sex', 'age', 'address']
assert list(cleaned_df.columns[:5]) == expected_first_columns
assert 'G3' in cleaned_df.columns
assert cleaned_df.isna().sum().sum() == 0
assert len(train_df) + len(valid_df) + len(test_df) == len(cleaned_df)

numeric_columns = ['age', 'Medu', 'Fedu', 'G1', 'G2', 'G3', 'absences']
print('Kiểu dữ liệu các cột số:')
display(cleaned_df[numeric_columns].dtypes.to_frame('dtype'))
print('Phân phối subject và nhóm đạt/trượt trong từng tập:')
for name, frame in {'train': train_df, 'valid': valid_df, 'test': test_df}.items():
    check = frame.assign(passed=frame['G3'] >= 10).groupby(['subject', 'passed']).size().rename('rows')
    print(f'\n{name}:')
    display(check.to_frame())

In [ ]:
# Kiểm tra không có dòng giống hệt nhau nằm ở nhiều tập.
def row_hashes(frame):
    return set(pd.util.hash_pandas_object(frame, index=False))

train_hashes = row_hashes(train_df)
valid_hashes = row_hashes(valid_df)
test_hashes = row_hashes(test_df)
overlap_counts = {
    'train_valid': len(train_hashes & valid_hashes),
    'train_test': len(train_hashes & test_hashes),
    'valid_test': len(valid_hashes & test_hashes)
}
display(pd.Series(overlap_counts, name='overlap_rows').to_frame())
assert all(value == 0 for value in overlap_counts.values())

summary = {
    'merged_rows': int(len(merged_df)),
    'cleaned_rows': int(len(cleaned_df)),
    'duplicate_rows_removed': int(duplicate_count),
    'train_rows': int(len(train_df)),
    'valid_rows': int(len(valid_df)),
    'test_rows': int(len(test_df)),
    'missing_values_after_cleaning': int(cleaned_df.isna().sum().sum()),
    'cross_split_exact_overlaps': overlap_counts
}
(PROJECT_ROOT / 'artifacts' / 'preprocessing_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Đã lưu preprocessing_summary.json.')

## Lưu ý

Notebook này chưa one-hot encoding hoặc scale toàn bộ dữ liệu. Các thao tác đó phải được fit trên `train.csv` trong notebook 04, sau đó dùng lại cho validation, test và dữ liệu API. Làm như vậy giúp kết quả đánh giá công bằng và tránh data leakage.